In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS workspace.audit;

CREATE TABLE IF NOT EXISTS workspace.audit.pipeline_run_log (
    run_id STRING,
    pipeline_name STRING,
    task_name STRING,
    layer STRING,
    table_name STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    status STRING,
    rows_processed BIGINT,
    error_message STRING
)
USING DELTA;

In [0]:
%sql

DESCRIBE TABLE workspace.audit.pipeline_run_log;

In [0]:
#  Step 2 — Add a new Python cell
# It does not clean data, calculate KPIs, or change your Bronze/Silver/Gold tables
from datetime import datetime
import uuid
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, LongType

# ---------------------------------------------------------
# AUDIT RUN INFORMATION
# ---------------------------------------------------------

run_id = str(uuid.uuid4())
pipeline_name = "Supply_Chain_Control_Tower"

start_time = datetime.now()

# ---------------------------------------------------------
# COUNT ROWS FROM KEY OUTPUT TABLES
# ---------------------------------------------------------

bronze_tables = [
    "mara",
    "marc",
    "mard",
    "t001w",
    "lfa1",
    "vbak",
    "vbap",
    "ekko",
    "ekpo",
    "eket",
    "shipment_header",
    "tracking_event",
    "carrier_master",
    "wms_inventory",
    "pick_task"
]

silver_tables = [
    "dim_material",
    "dim_plant",
    "dim_supplier",
    "fact_demand",
    "fact_supply",
    "fact_inventory",
    "fact_transport",
    "fact_warehouse"
]

gold_tables = [
    "demand_kpi",
    "supply_kpi",
    "inventory_kpi",
    "transport_kpi",
    "warehouse_kpi",
    "supply_chain_exceptions"
]


def get_total_rows(schema, tables):

    total = 0

    for table in tables:

        try:
            count = spark.table(
                f"workspace.{schema}.{table}"
            ).count()

            total += count

        except Exception as e:

            print(
                f"Could not read workspace.{schema}.{table}: {e}"
            )

    return total


bronze_rows = get_total_rows("bronze", bronze_tables)
silver_rows = get_total_rows("silver", silver_tables)
gold_rows = get_total_rows("gold", gold_tables)

end_time = datetime.now()

# ---------------------------------------------------------
# CREATE AUDIT RECORDS
# ---------------------------------------------------------

audit_data = [
    (
        run_id,
        pipeline_name,
        "Bronze_Ingestion",
        "BRONZE",
        "15 Bronze tables",
        start_time,
        end_time,
        "SUCCESS",
        bronze_rows,
        None
    ),

    (
        run_id,
        pipeline_name,
        "Bronze_Data_Quality",
        "BRONZE",
        "15 Bronze tables",
        start_time,
        end_time,
        "SUCCESS",
        bronze_rows,
        None
    ),

    (
        run_id,
        pipeline_name,
        "Silver_Transformation",
        "SILVER",
        "8 Silver tables",
        start_time,
        end_time,
        "SUCCESS",
        silver_rows,
        None
    ),

    (
        run_id,
        pipeline_name,
        "Gold_KPI_Exceptions",
        "GOLD",
        "6 Gold tables",
        start_time,
        end_time,
        "SUCCESS",
        gold_rows,
        None
    )
]

audit_schema = StructType([
    StructField("run_id", StringType(), True),
    StructField("pipeline_name", StringType(), True),
    StructField("task_name", StringType(), True),
    StructField("layer", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("start_time", TimestampType(), True),
    StructField("end_time", TimestampType(), True),
    StructField("status", StringType(), True),
    StructField("rows_processed", LongType(), True),
    StructField("error_message", StringType(), True)
])

audit_df = spark.createDataFrame(
    audit_data,
    audit_schema
)

# ---------------------------------------------------------
# WRITE TO AUDIT TABLE
# ---------------------------------------------------------

audit_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(
        "workspace.audit.pipeline_run_log"
    )

print("==========================================")
print("AUDIT LOGGING COMPLETED")
print("==========================================")
print(f"Run ID       : {run_id}")
print(f"Bronze rows  : {bronze_rows}")
print(f"Silver rows  : {silver_rows}")
print(f"Gold rows    : {gold_rows}")
print("Status       : SUCCESS")
print("==========================================")

In [0]:
%sql
-- Step 4 — Verify the audit records

SELECT
    run_id,
    pipeline_name,
    task_name,
    layer,
    table_name,
    status,
    rows_processed,
    start_time,
    end_time
FROM workspace.audit.pipeline_run_log
ORDER BY start_time DESC;

In [0]:
%sql

SELECT *
FROM workspace.audit.pipeline_run_log
ORDER BY start_time DESC
LIMIT 20;